In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import joblib # <--- PERBAIKAN: Import yang hilang
warnings.filterwarnings("ignore")

# Import dari Sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    precision_recall_curve, f1_score
)
from sklearn.ensemble import IsolationForest

# Setup Device (CUDA atau CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"Menggunakan device: {device} ({torch.cuda.get_device_name(0)})")
else:
    print(f"Menggunakan device: {device}.")

# --- KONFIGURASI UTAMA ---
# (Sesuaikan path ini dengan struktur folder Anda)
DATA_PATH = '../1_Data/' 
MODEL_PATH = '../3_Output/'
os.makedirs(MODEL_PATH, exist_ok=True)

# Konstanta Model
RANDOM_SEED = 42
TIME_STEPS = 10 # Panjang sekuens untuk LSTM
BATCH_SIZE = 128
N_EPOCHS = 30 # Jumlah epoch (bisa dinaikkan ke 50-100 untuk hasil lebih baik)

# Set random seed untuk reproduktibilitas
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

print("Libraries imported and setup complete.")

SMOTE dan XGBoost berhasil di-import.
Menggunakan device: cuda (NVIDIA GeForce RTX 3050)
Libraries imported and setup complete.


In [2]:
try:
    df_train = pd.read_csv(os.path.join(DATA_PATH, 'train_v3.csv'))
    df_val = pd.read_csv(os.path.join(DATA_PATH, 'val_v3.csv'))
    df_test = pd.read_csv(os.path.join(DATA_PATH, 'test_v3.csv'))
    print("Dataset V3 (Train, Val, Test) berhasil di-load.")
    print(f"Train shape: {df_train.shape}")
    print(f"Val shape: {df_val.shape}")
    print(f"Test shape: {df_test.shape}")
except FileNotFoundError:
    print(f"Error: Pastikan file 'train_v3.csv', 'val_v3.csv', 'test_v3.csv' ada di {DATA_PATH}")
    raise

Dataset Kaggle Fraud berhasil di-load. Shape: (284807, 31)
Data berhasil di-sort berdasarkan 'Time'.
Data Normal untuk Training AE: (198980, 31)
Data Full untuk Validasi: (42721, 31)
Data Full untuk Test: (42722, 31)


In [9]:
print("\n--- [Evaluasi Final] Mengevaluasi Model Usulan (AE Statis) ---")

# 1. Dapatkan Error & Label dari DATA TES (CAMPURAN)
test_errors_statis_mixed = []
test_labels_statis_mixed = []
with torch.no_grad():
    for (data, labels) in test_loader_statis_mixed:
        data = data.to(device)
        reconstructions = model_ae_statis(data)
        errors = torch.mean((data - reconstructions) ** 2, dim=1).cpu().numpy()
        test_errors_statis_mixed.extend(errors)
        test_labels_statis_mixed.extend(labels.numpy())

test_errors_statis_mixed = np.array(test_errors_statis_mixed)
test_labels_statis_mixed = np.array(test_labels_statis_mixed)

# 2. Terapkan Threshold Terbaik (dari Sel 7)
y_pred_final = [1 if e > best_threshold else 0 for e in test_errors_statis_mixed]
y_true_final = test_labels_statis_mixed

# 3. Tampilkan Hasil
print("\n--- Hasil Model Usulan (AE Statis Optimized) (Tabel 1) ---")
print(classification_report(y_true_final, y_pred_final, target_names=['Normal (0)', 'Fraud (1)']))
print("CATATAN: Bandingkan F1-Score (Fraud) ini dengan semua baseline.")

# 4. (Opsional) Analisis Sensitivitas P99 (Untuk perbandingan)
threshold_p99 = np.percentile(val_errors_statis_mixed, 99) 
y_pred_p99 = [1 if e > threshold_p99 else 0 for e in test_errors_statis_mixed]
print("\n--- Perbandingan: Hasil jika menggunakan P99 (Lama) ---")
print(classification_report(y_true_final, y_pred_p99, target_names=['Normal (0)', 'Fraud (1)']))

print("\nNotebook 2 V5 (Strategi Baru) Selesai!")



--- [Evaluasi Final] Mengevaluasi Model Usulan (AE Statis) ---

--- Hasil Model Usulan (AE Statis Optimized) (Tabel 1) ---
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00     42670
   Fraud (1)       0.61      0.42      0.50        52

    accuracy                           1.00     42722
   macro avg       0.81      0.71      0.75     42722
weighted avg       1.00      1.00      1.00     42722

CATATAN: Bandingkan F1-Score (Fraud) ini dengan semua baseline.

--- Perbandingan: Hasil jika menggunakan P99 (Lama) ---
              precision    recall  f1-score   support

  Normal (0)       1.00      0.99      1.00     42670
   Fraud (1)       0.14      0.77      0.24        52

    accuracy                           0.99     42722
   macro avg       0.57      0.88      0.62     42722
weighted avg       1.00      0.99      1.00     42722


Notebook 2 V5 (Strategi Baru) Selesai!
